# Module 05 — Notebook 4: Eval Plots Mini-Project

## What You'll Build

A complete visualization pipeline that saves a set of publication-ready plots to `output/plots/`:

1. `model_bars.png` — average score per model
2. `task_bars.png` — average score per task
3. `scorecard_heatmap.png` — full model × task heatmap
4. `score_distribution.png` — histogram of all scores
5. `summary_4panel.png` — 2×2 summary figure

**Time:** ~25 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json
from pathlib import Path

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

CSV_PATH  = Path("../../data/synthetic/evaluation_results.csv")
JSON_PATH = Path("../../data/synthetic/model_outputs.json")
OUTPUT_DIR = Path("../../output/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
with open(JSON_PATH) as f:
    outputs_df = pd.DataFrame(json.load(f))
outputs_df["response_length"] = outputs_df["response"].str.len()

print("Output dir:", OUTPUT_DIR.resolve())
print("eval_df:", df.shape, "  outputs_df:", outputs_df.shape)

## Step 1: Model Bar Chart

A bar chart comparing mean scores across models — the first thing you'd put in an eval report.

In [ ]:
per_model = df.groupby("model")["score"].mean().sort_values(ascending=False)
colors = ["#2196F3" if "a" in m else "#FF5722" for m in per_model.index]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(per_model.index, per_model.values, color=colors, alpha=0.85, edgecolor="white")

for bar, val in zip(bars, per_model.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10)

ax.axhline(df["score"].mean(), color="black", linestyle="--", linewidth=1,
           label=f"Overall mean ({df['score'].mean():.3f})")
ax.set_title("Mean Evaluation Score by Model", fontsize=13, fontweight="bold")
ax.set_ylabel("Mean Score")
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "model_bars.png", dpi=150, bbox_inches="tight")
print("Saved model_bars.png")
plt.show()

## Step 2: Task Bar Chart

Horizontal bars for tasks — easier to read when labels are long.

In [ ]:
per_task = df.groupby("task")["score"].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(per_task.index, per_task.values, color="steelblue", alpha=0.85)

for i, val in enumerate(per_task.values):
    ax.text(val + 0.005, i, f"{val:.3f}", va="center", fontsize=9)

ax.axvline(per_task.mean(), color="crimson", linestyle="--", linewidth=1,
           label=f"Mean ({per_task.mean():.3f})")
ax.set_title("Mean Score by Task", fontsize=13, fontweight="bold")
ax.set_xlabel("Mean Score")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)
ax.legend()
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "task_bars.png", dpi=150, bbox_inches="tight")
print("Saved task_bars.png")
plt.show()

## Step 3: Scorecard Heatmap

The model × task scorecard — standard format for benchmark reports.

In [ ]:
pivot = df.pivot_table(index="model", columns="task", values="score")

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    pivot, annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=0.5, vmax=1.0,
    linewidths=0.5, ax=ax
)

# Flag critical failures
for row_i, model in enumerate(pivot.index):
    for col_i, task in enumerate(pivot.columns):
        if pivot.loc[model, task] < 0.70:
            ax.text(col_i + 0.5, row_i + 0.15, "⚠",
                    ha="center", va="center", color="black", fontsize=14)

ax.set_title("Model × Task Scorecard (⚠ = below 0.70)", fontsize=12, pad=12)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "scorecard_heatmap.png", dpi=150, bbox_inches="tight")
print("Saved scorecard_heatmap.png")
plt.show()

## Step 4: Score Distribution Histogram

A histogram split by model shows whether score distributions overlap or are clearly separated.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for model, color in zip(df["model"].unique(), ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]):
    model_scores = df[df["model"] == model]["score"]
    ax.hist(model_scores, bins=5, alpha=0.5, label=model, color=color, edgecolor="white")

ax.axvline(df["score"].mean(), color="black", linestyle="--", linewidth=1.2, label="Overall mean")
ax.set_title("Score Distribution by Model", fontsize=13)
ax.set_xlabel("Score")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
print("Saved score_distribution.png")
plt.show()

---
## Your Turn — Exercise 1: Compute the Scorecard Data

Before building the 4-panel summary figure, prepare the data.

1. Store per-model mean scores in `per_model` (a Series, sorted descending).
2. Store per-task mean scores in `per_task` (a Series, sorted ascending).
3. Build the pivot table in `scorecard` (model as index, task as columns, score as values).
4. Store the overall mean score in `overall_mean`, rounded to 4 decimal places.

In [ ]:
# YOUR CODE HERE
per_model    = None   # Series: model → mean score, sorted descending
per_task     = None   # Series: task → mean score, sorted ascending
scorecard    = None   # DataFrame: pivot_table(index="model", columns="task")
overall_mean = None   # float, rounded to 4 decimal places

In [ ]:
check_type(per_model, pd.Series, "per_model is a Series")
check_type(per_task, pd.Series, "per_task is a Series")
check_type(scorecard, pd.DataFrame, "scorecard is a DataFrame")
check_equal(scorecard.shape, (4, 5), "scorecard is 4×5")
check_approx(overall_mean, 0.8225, 1e-4, "overall_mean")

---
## Your Turn — Exercise 2: Build and Save the 4-Panel Summary

Using the data you computed in Exercise 1, build a 2×2 summary figure and save it.

- `axes[0, 0]`: bar chart of per-model mean scores
- `axes[0, 1]`: horizontal bar chart of per-task mean scores
- `axes[1, 0]`: seaborn box plot by model
- `axes[1, 1]`: heatmap of `scorecard`

Save to `OUTPUT_DIR / "summary_4panel.png"` and store the save path in `summary_path`.

In [ ]:
# YOUR CODE HERE
summary_path = None   # Path object — OUTPUT_DIR / "summary_4panel.png"

# fig, axes = plt.subplots(2, 2, figsize=(14, 9))
#
# axes[0, 0]: per-model bar
# axes[0, 1]: per-task barh
# axes[1, 0]: sns.boxplot by model
# axes[1, 1]: sns.heatmap(scorecard, ...)
#
# fig.suptitle("Evaluation Summary", fontsize=14)
# plt.tight_layout()
# fig.savefig(summary_path, dpi=150, bbox_inches="tight")
# plt.show()

In [ ]:
check_type(summary_path, Path, "summary_path is a Path")
check_equal(summary_path.exists(), True, "summary_4panel.png was saved to disk")

---
## Your Turn — Exercise 3: Flag Rate Bar Chart

Using `outputs_df`, create and save a bar chart showing the **flag rate per model**.

1. Compute `flag_rates` — a Series of flag rate (fraction flagged) per model, sorted descending.
2. Create a bar chart with a horizontal reference line at `0.5` ("50% flagged").
3. Save to `OUTPUT_DIR / "flag_rates.png"` and store the path in `flag_path`.

In [ ]:
# YOUR CODE HERE
flag_rates = None   # Series: model → flag rate, sorted descending
flag_path  = None   # Path to saved file

In [ ]:
check_type(flag_rates, pd.Series, "flag_rates is a Series")
# model-a-v1 has 0 flagged outputs → flag rate 0.0
check_approx(float(flag_rates["model-a-v1"]), 0.0, 1e-6, "model-a-v1 flag rate is 0")
check_type(flag_path, Path, "flag_path is a Path")
check_equal(flag_path.exists(), True, "flag_rates.png was saved to disk")

---
## Your Turn — Exercise 4: List All Saved Files

Store the names (not full paths) of all `.png` files now in `OUTPUT_DIR` in a sorted list
called `saved_plots`.

> **Hint:** `sorted([p.name for p in OUTPUT_DIR.glob("*.png")])`

In [ ]:
# YOUR CODE HERE
saved_plots = None   # sorted list of .png filenames in OUTPUT_DIR
print("Saved plots:", saved_plots)

In [ ]:
check_type(saved_plots, list, "saved_plots is a list")
check_contains(saved_plots, "flag_rates.png", "flag_rates.png is in saved_plots")
check_contains(saved_plots, "summary_4panel.png", "summary_4panel.png is in saved_plots")
check_contains(saved_plots, "scorecard_heatmap.png", "scorecard_heatmap.png is in saved_plots")

---
## Why This Matters for AI Research Engineering

An automated plotting pipeline — load data, compute statistics, generate figures, save to a folder — is what you build when you want evaluation results to update automatically every time a new model runs.

The pattern you've practiced here:
```
data → compute → plot → save → verify
```
is the same pattern used in production eval pipelines at AI labs. The `check_equal(path.exists(), True)` pattern is the notebook equivalent of an assertion in a CI test — it confirms the pipeline actually produced output, not just ran without error.

The flag rate chart is a safety-specific visualization you'll use constantly: which model is producing problematic outputs, at what rate, and is that rate above or below some threshold? That single bar chart, in a weekly memo, can trigger a deployment decision.

## Summary — Module 05 Complete

| Topic | Key patterns |
|-------|--------------|
| **Figure/Axes** | `fig, ax = plt.subplots(figsize=(w, h))` |
| **Bar charts** | `ax.bar()`, `ax.barh()`, value labels, reference lines |
| **Histograms** | `ax.hist()`, `sns.histplot(kde=True)` |
| **Box/strip plots** | `sns.boxplot()` + `sns.stripplot()` overlay |
| **Scatter** | `ax.scatter()`, `sns.stripplot(hue=...)` |
| **Heatmaps** | `sns.heatmap(pivot, annot=True, cmap="RdYlGn")` |
| **Subplots** | `plt.subplots(nrows, ncols)`, `axes[r, c]` |
| **Saving** | `fig.savefig(path, dpi=150, bbox_inches="tight")` |

**Next:** Module 06 — Writing Reusable Scripts, where you'll turn notebook analysis into command-line tools.